# 🔮 Brujas de Datos: La Apertura
### She Data · Charla 1 · con Lina Serna

Hoy no vamos a hablar de teoría. Vamos a **abrir un dataset real** y ver, paso a paso, cómo trabajamos los datos en She Data: de la curiosidad al insight.

El dataset: los **152 juicios de brujería de Salem (1692)** — compilado por el historiador Richard Latner (Tulane University). Sí, leíste bien: vamos a investigar a las *brujas originales* con las herramientas de las *brujas de datos* de hoy. 🪄✨

**Agenda de la charla:**
1. Cargar y explorar los datos (¿con qué estamos trabajando?)
2. Limpieza rápida (los datos nunca llegan perfectos)
3. Preguntas que le hacemos a los datos
4. Visualizaciones que cuentan la historia
5. Una mini-predicción para cerrar con magia


## 1️⃣ Cargar los datos

Lo primero siempre: **mirar los datos crudos antes de asumir nada.**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Si corres esto en Google Colab, sube el archivo brujas_acusadas_salem_1692.csv
# o monta tu Google Drive. En local, ajusta la ruta.
df = pd.read_csv("brujas_acusadas_salem_1692.csv")

print(f"Filas: {df.shape[0]}  |  Columnas: {df.shape[1]}")
df.head(10)


**Contexto rápido de las columnas:**
- `nombre`: persona acusada formalmente de brujería
- `residencia`: pueblo donde vivía al momento de la acusación
- `mes_acusacion`: mes de 1692 en que fue acusada
- `mes_ejecucion`: mes en que fue ejecutada (vacío si no lo fue)
- `ejecutada`: Sí / No

Dato curioso: de las 152 personas acusadas, la mayoría **nunca fue ejecutada**. Vamos a comprobarlo con datos, no con la fama que tiene Salem.

## 2️⃣ Limpieza rápida: la regla de oro

> *"Los datos nunca llegan limpios. Si parecen perfectos, sospecha."*

Revisamos nulos, tipos y valores raros antes de confiar en nada.

In [ ]:
df.info()
print("\nValores nulos por columna:")
print(df.isna().sum())


In [ ]:
# ¿Cuántas acusaciones tienen mes desconocido?
sin_mes = df["mes_acusacion"].eq("Desconocido").sum()
print(f"Acusaciones con mes desconocido: {sin_mes} de {len(df)}")

# Esto es importante: en los registros originales, el mes '-1' significa
# 'no se sabe con certeza'. Lo dejamos explícito en vez de inventarlo.


## 3️⃣ Las preguntas que le hacemos a los datos

Antes de graficar cualquier cosa, definimos **qué queremos saber**:

- ¿De dónde eran las personas acusadas? ¿Hay un pueblo que concentra el pánico?
- ¿En qué mes se disparó la histeria?
- ¿Qué proporción de acusadas terminó ejecutada?

### 🏘️ ¿De dónde eran las acusadas?

In [ ]:
top_pueblos = df["residencia"].value_counts().head(10)
top_pueblos


In [ ]:
plt.style.use("dark_background")
fig, ax = plt.subplots(figsize=(9, 5.5))

colors = ["#c88bff" if v != top_pueblos.max() else "#ff4fa3" for v in top_pueblos.values]
ax.barh(top_pueblos.index[::-1], top_pueblos.values[::-1], color=colors[::-1])

ax.set_title("¿De dónde eran las 'brujas' acusadas en 1692?", fontsize=14, fontweight="bold", color="white")
ax.set_xlabel("Número de personas acusadas")
for i, v in enumerate(top_pueblos.values[::-1]):
    ax.text(v + 0.5, i, str(v), va="center", color="white", fontsize=10)

plt.tight_layout()
plt.show()


💡 **Insight:** Andover concentra casi un tercio de todas las acusaciones — más que Salem Town y Salem Village juntos. El pánico de 1692 no fue solo un fenómeno de "Salem"; fue una ola regional. Este tipo de hallazgo es exactamente lo que buscamos: **el dato que contradice la versión popular de la historia.**

### 📅 ¿Cuándo se disparó la histeria?

In [ ]:
orden_meses = ["Febrero","Marzo","Abril","Mayo","Junio","Julio",
               "Agosto","Septiembre","Octubre","Noviembre"]

por_mes = (df[df["mes_acusacion"] != "Desconocido"]
           ["mes_acusacion"].value_counts()
           .reindex(orden_meses))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(por_mes.index, por_mes.values, marker="o", markersize=9,
        color="#c88bff", linewidth=2.5)
ax.fill_between(range(len(por_mes)), por_mes.values, color="#c88bff", alpha=0.15)

ax.set_title("Acusaciones de brujería por mes · Salem, 1692", fontsize=14, fontweight="bold", color="white")
ax.set_ylabel("Número de acusaciones")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


💡 **Insight:** el pico está en abril-mayo, justo después de que empezaran los primeros juicios formales en el Tribunal de Oyer and Terminer (verano de 1692). La curva sube, se mantiene alta y cae en otoño — cuando el propio gobernador disolvió ese tribunal por las críticas a sus métodos. **Los datos muestran el efecto de una decisión institucional**, no solo "histeria colectiva" sin causa.

### ⚖️ ¿Qué proporción terminó ejecutada?

In [ ]:
conteo = df["ejecutada"].value_counts()
porcentajes = (conteo / conteo.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(6, 6))
colors = ["#3a3a3a", "#ff4fa3"]
wedges, texts, autotexts = ax.pie(
    conteo.values,
    labels=[f"No ejecutada\n({conteo['No']})", f"Ejecutada\n({conteo['Si']})"],
    autopct="%1.1f%%",
    colors=colors,
    startangle=90,
    textprops={"color": "white", "fontsize": 11}
)
ax.set_title("De 152 acusadas, ¿cuántas fueron ejecutadas?", fontsize=13, fontweight="bold", color="white")
plt.tight_layout()
plt.show()


💡 **Insight:** solo el **13%** de las personas acusadas fue ejecutada. El 87% restante fue encarcelada, esperó juicio, y en muchos casos fue liberada cuando el nuevo tribunal (enero 1693) cambió de criterio. La narrativa popular de "Salem = ejecuciones masivas" no coincide con el dato real — y esto es justo el tipo de verificación que hacemos todo el tiempo como analistas: **cuestionar el relato con la evidencia.**

## 4️⃣ Cruzando variables: lugar + tiempo

Una sola variable rara vez cuenta la historia completa. Cruzamos pueblo y mes para ver el patrón de propagación.

In [ ]:
top5_pueblos = df["residencia"].value_counts().head(5).index
sub = df[df["residencia"].isin(top5_pueblos) & (df["mes_acusacion"] != "Desconocido")]

tabla = pd.crosstab(sub["residencia"], sub["mes_acusacion"])
tabla = tabla[[c for c in orden_meses if c in tabla.columns]]

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(tabla.values, cmap="RdPu", aspect="auto")

ax.set_xticks(range(len(tabla.columns)))
ax.set_xticklabels(tabla.columns, rotation=30, ha="right")
ax.set_yticks(range(len(tabla.index)))
ax.set_yticklabels(tabla.index)
ax.set_title("Propagación del pánico: pueblo × mes", fontsize=13, fontweight="bold", color="white")

for i in range(len(tabla.index)):
    for j in range(len(tabla.columns)):
        val = tabla.values[i, j]
        if val > 0:
            ax.text(j, i, int(val), ha="center", va="center",
                    color="white" if val < tabla.values.max()/2 else "black", fontsize=9)

plt.colorbar(im, ax=ax, label="N° de acusaciones")
plt.tight_layout()
plt.show()


💡 **Insight:** Salem Village enciende la mecha en marzo-abril. Andover se incendia después, en julio-septiembre — meses más tarde, y con muchas más acusaciones. El "contagio social" del pánico es visible en los propios datos: **se mueve de un epicentro hacia afuera, con retraso.**

## 5️⃣ Cierre con magia: ¿se podía predecir quién sería acusada?

Esta es la pregunta de inspiración original del dataset: *¿se puede predecir la acusación a partir de otras variables?*

Con este dataset no tenemos variables socioeconómicas por persona (eso vive en otro de los datasets de Latner: impuestos y peticiones). Pero sí podemos mostrar, en vivo, **cómo se vería el primer paso de un modelo**: convertir "residencia" en una variable que un modelo pueda usar.

In [ ]:
# Ejemplo didáctico: one-hot encoding de la residencia
# (el primer paso real antes de entrenar cualquier modelo)
demo = pd.get_dummies(df[["nombre", "residencia", "ejecutada"]], columns=["residencia"])
demo.head()


Con esto —y el dataset de impuestos/peticiones políticas de Salem Village— se podría entrenar un modelo simple de clasificación para explorar esa pregunta. **Queda de tarea/reto para quien quiera seguir jugando con el dataset completo** (link abajo). 🔮

---

### 🪄 Lo que vimos hoy, en resumen
1. Cargar y mirar los datos crudos primero.
2. Nunca asumir que están limpios — revisar nulos y valores raros.
3. Hacer preguntas concretas antes de graficar.
4. Una visualización simple puede desmentir un mito de 300 años.
5. Cruzar variables revela patrones que una sola variable esconde.

### 📚 Fuente y créditos
Dataset: *Salem Witchcraft Dataset*, compilado por el Prof. Richard Latner (Tulane University).
Úsalo, compártelo y adáptalo citando su trabajo.

### ✨ Síguenos
**She Data** — ninguna mujer aprende datos sola.
